In [ ]:
# ═══ Job 재설정: SQL Runner 기반 (4개 Job 전체 구성) ═══
import requests, json

host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
base_url = f"https://{host}/api/2.1/jobs"

RUNNER = "/Workspace/Users/seonghwan.park@stradvision.com/.sql/_run_sql_query"
SQL_BASE = "/Workspace/Users/seonghwan.park@stradvision.com/.sql"
alert_email = "seonghwan.park@stradvision.com"

email_config = {
    "on_failure": [alert_email],
    "on_success": [alert_email],
    "no_alert_for_skipped_runs": True
}

# --- Job 1: kpi_staging_daily (794781503662993) ---
# stg 4종 + dim 3종 = 병렬
# int_object_counts → stg_objects 완료 후
# int_command_slots → stg_workspace_commands 완료 후
resp = requests.post(f"{base_url}/reset", headers=headers, json={
    "job_id": 794781503662993,
    "new_settings": {
        "name": "kpi_staging_daily",
        "schedule": {"quartz_cron_expression": "0 0 4 * * ?", "timezone_id": "UTC", "pause_status": "UNPAUSED"},
        "email_notifications": email_config,
        "tasks": [
            {"task_key": "stg_tasks", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/staging/stg__tasks.dbquery.ipynb"}
            }},
            {"task_key": "stg_workspace_commands", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/staging/stg__workspace_commands.dbquery.ipynb"}
            }},
            {"task_key": "stg_task_transition_events", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/staging/stg__task_transition_events.dbquery.ipynb"}
            }},
            {"task_key": "stg_objects", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/staging/stg__objects.dbquery.ipynb"}
            }},
            {"task_key": "dim_companies", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/dim__companies.dbquery.ipynb"}
            }},
            {"task_key": "dim_assignments", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/dim__assignments.dbquery.ipynb"}
            }},
            {"task_key": "dim_policies", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/dim__policies.dbquery.ipynb"}
            }},
            {"task_key": "int_object_counts", "depends_on": [{"task_key": "stg_objects"}], "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/int__object_counts_by_task.dbquery.ipynb"}
            }},
            {"task_key": "int_command_slots", "depends_on": [{"task_key": "stg_workspace_commands"}], "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/int__command_slots_by_task.dbquery.ipynb"}
            }}
        ]
    }
})
print(f"Job 1 (kpi_staging_daily) reset: {resp.status_code}")
if resp.status_code != 200:
    print(resp.text)

# --- Job 2: focus_drop_daily (11361746396560) ---
resp = requests.post(f"{base_url}/reset", headers=headers, json={
    "job_id": 11361746396560,
    "new_settings": {
        "name": "focus_drop_daily",
        "schedule": {"quartz_cron_expression": "0 30 4 * * ?", "timezone_id": "UTC", "pause_status": "UNPAUSED"},
        "email_notifications": email_config,
        "tasks": [
            {"task_key": "session_metrics", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/int__focus_drop_session_metrics.dbquery.ipynb"}
            }},
            {"task_key": "task_idle_rollup", "depends_on": [{"task_key": "session_metrics"}], "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/int__focus_drop_task_idle_rollup.dbquery.ipynb"}
            }},
            {"task_key": "session_tags", "depends_on": [{"task_key": "session_metrics"}], "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/int__focus_drop_session_tags.dbquery.ipynb"}
            }},
            {"task_key": "user_day_kpi", "depends_on": [{"task_key": "session_tags"}], "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/intermediate/int__focus_drop_user_day_kpi.dbquery.ipynb"}
            }}
        ]
    }
})
print(f"Job 2 (focus_drop_daily) reset: {resp.status_code}")

# --- Job 3: focus_drop_weekly (1008497285809413) ---
resp = requests.post(f"{base_url}/reset", headers=headers, json={
    "job_id": 1008497285809413,
    "new_settings": {
        "name": "focus_drop_weekly",
        "schedule": {"quartz_cron_expression": "0 0 3 ? * MON", "timezone_id": "UTC", "pause_status": "UNPAUSED"},
        "email_notifications": email_config,
        "tasks": [
            {"task_key": "session_thresholds", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {
                    "sql_path": f"{SQL_BASE}/intermediate/int__focus_drop_session_thresholds.dbquery.ipynb",
                    "rolling_window_days": "30", "is_bootstrap": "false"
                }
            }},
            {"task_key": "user_thresholds", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {
                    "sql_path": f"{SQL_BASE}/intermediate/int__focus_drop_user_thresholds.dbquery.ipynb",
                    "rolling_window_days": "30", "is_bootstrap": "false"
                }
            }}
        ]
    }
})
print(f"Job 3 (focus_drop_weekly) reset: {resp.status_code}")

# --- Job 4: kpi_weekly (1034375050896341) ---
resp = requests.post(f"{base_url}/reset", headers=headers, json={
    "job_id": 1034375050896341,
    "new_settings": {
        "name": "kpi_weekly",
        "schedule": {"quartz_cron_expression": "0 0 5 ? * MON", "timezone_id": "UTC", "pause_status": "UNPAUSED"},
        "email_notifications": email_config,
        "tasks": [
            {"task_key": "production_volume_weekly", "notebook_task": {
                "notebook_path": RUNNER,
                "base_parameters": {"sql_path": f"{SQL_BASE}/mart/mrt__production_volume_weekly.dbquery.ipynb"}
            }}
        ]
    }
})
print(f"Job 4 (kpi_weekly) reset: {resp.status_code}")

# --- 검증 ---
print("\n" + "="*60)
print("📋 최종 확인")
print("="*60)
for jid, expected_name in [(794781503662993,"kpi_staging_daily"),(11361746396560,"focus_drop_daily"),(1008497285809413,"focus_drop_weekly"),(1034375050896341,"kpi_weekly")]:
    resp = requests.get(f"{base_url}/get?job_id={jid}", headers=headers)
    if resp.status_code == 200:
        j = resp.json().get("settings", {})
        actual_name = j.get("name", "???")
        tasks = j.get("tasks", [])
        schedule = j.get("schedule", {}).get("quartz_cron_expression", "N/A")
        icon = "✅" if actual_name == expected_name else "❌"
        print(f"  {icon} {actual_name} — {len(tasks)} tasks, schedule: {schedule}")
        for t in tasks:
            deps = t.get("depends_on", [])
            dep_str = f" [← {','.join(d['task_key'] for d in deps)}]" if deps else ""
            params = t.get("notebook_task", {}).get("base_parameters", {})
            sql_path = params.get("sql_path", "")
            print(f"      {t['task_key']}{dep_str}: ...{sql_path[-55:]}")